In [ ]:
from datetime import datetime,timedelta
from jqdata import get_trade_days
from tqdm import tqdm
from collections import deque
import csv
import os

In [ ]:
indexes = get_all_securities(types=['index'], date=None)

In [ ]:
indexes[indexes.display_name.str.contains("中证全指")]

In [ ]:
index_code = '000985.XSHG'

In [ ]:
def get_resume_date(csv_path='breadth.csv'):
    try:
        with open(csv_path, 'r', encoding='utf-8-sig') as f:
            next(f)
            last_line = deque(f, maxlen=1)[0]
        return last_line.strip().split(',')[0]
    except (FileNotFoundError, StopIteration, IndexError):
        return None     # 文件不存在或只有表头,从头跑

In [ ]:
last = get_resume_date()
start_date = pd.Timestamp(last) + pd.Timedelta(days=1) if last else indexes.loc[index_code].start_date
start_date

In [ ]:
end_date = start_date + timedelta(days=1000)
if end_date > datetime.now() - timedelta(days=1):
    end_date = datetime.now() - timedelta(days=1)
end_date

In [ ]:
# 获取交易天数
trade_days = get_trade_days(start_date=start_date, end_date=end_date)
len(trade_days),trade_days[-1]

In [ ]:
csv_path = 'breadth.csv'
write_header = not os.path.exists(csv_path)
with open(csv_path, 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['date', 'up', 'down', 'flat', 'total'])
    if write_header:
        writer.writeheader()
    for d in tqdm(trade_days, desc="计算每日宽度"):
        d_str = pd.Timestamp(d).strftime('%Y-%m-%d')
        # 1) 当日成分股
        codes = get_index_stocks(index_code, date=d_str)
        if not codes:
            continue
            
        # 2) 当日 + 前一交易日的量价
        df = get_price(
            codes,
            end_date=d_str,
            frequency='daily',
            skip_paused=False,
            fq='pre',
            count=2,
            panel=False,
            fill_paused=True,
        )
        if df.empty:
            continue
            
        # 3) 长表 -> 宽表 (一行一只股票,一列一个交易日)
        close_wide = df.pivot(index='code', columns='time', values='close')
        close_wide = close_wide.dropna(how='any')          # 排除新股/无前一交易日
        if close_wide.shape[1] < 2:
            continue
        
        close_wide = close_wide[sorted(close_wide.columns)]
        prev_day, today = close_wide.columns
        
        up   = (close_wide[today] >  close_wide[prev_day]).sum()
        down = (close_wide[today] <  close_wide[prev_day]).sum()
        flat = (close_wide[today] == close_wide[prev_day]).sum()
        
        writer.writerow({
            'date':  d,
            'up':    up,
            'down':  down,
            'flat':  flat,
            'total': up + down + flat,
        })
        f.flush()